# Creativity Experiment: Comprehensive Torrance-Style Test with Goodfire Steering

This notebook implements a comprehensive divergent thinking experiment inspired by the Torrance Test.
We test creativity across multiple dimensions with 7 different tasks:
1. **Alternate Uses**: Creative ways to use a brick (divergent thinking)
2. **Implications**: Consequences if animals could talk (consequence thinking)
3. **Imagination**: What you'd do if you could fly (personal creativity)
4. **Problem Solving**: Gaining knowledge without books (adaptive thinking)
5. **Creative Planning**: Road trip vs tree house (creative choice)
6. **Product Improvement**: Enhancing a stapler (innovative thinking)
7. **Creative Writing**: Story about a mysterious door (narrative creativity)

For each task, we compare:
- **Baseline**: Normal responses
- **Steering**: Responses with creativity-enhancing neural interventions via Goodfire

We then analyze feature activations to understand how creativity manifests differently across task types.

In [1]:
# Clear cache for fresh responses
import shutil
import os

cache_dir = '.edsl_cache'
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

In [2]:
from edsl import QuestionFreeText, QuestionList, QuestionMultipleChoice, Survey, Agent, AgentList, Model
import os
import pandas as pd
import goodfire

In [3]:
# Initialize Goodfire
client = goodfire.Client(os.getenv("GOODFIRE_API_KEY"))
base_variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

In [4]:
# Simplified Creativity Experiment Class
class CreativityExperiment:
    def __init__(self, agents, model):
        self.agents = agents
        self.model = model
        self.questions = []
        self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

    def setup(self):
        # Multiple creativity tasks for comprehensive divergent thinking assessment
        self.questions = [
            # Task 1: Classic divergent thinking - alternate uses
            QuestionList(
                question_name="brick_uses",
                question_text="List very detailed creative and diverse ways you can use a brick. Each answer should be a paragraph.",
                max_list_items=10
            ),
            
            # Task 2: Implications and consequences
            QuestionFreeText(
                question_name="animals_talk",
                question_text="What would be the implications if animals could talk? Think broadly and creatively about social, environmental, ethical, and practical consequences."
            ),
            
            # Task 3: Personal imagination - possibilities
            QuestionList(
                question_name="flying_ability",
                question_text="Just suppose you woke up one morning and found you could fly. What would you do? List as many things as you can think of.",
                max_list_items=15
            ),
            
            # Task 4: Problem-solving creativity
            QuestionFreeText(
                question_name="books_disappear",
                question_text="If all books were to disappear, how would you gain knowledge? Describe creative and diverse methods in detail."
            ),
            
            # Task 5: Creative planning choice
            QuestionMultipleChoice(
                question_name="planning_task",
                question_text="Which task would allow you to be more creative and why?",
                question_options=["Organizing a cross-country road trip", "Building a tree house"]
            ),
            
            # Task 6: Product improvement - detailed enhancements
            QuestionList(
                question_name="stapler_improvements",
                question_text="Your goal is to improve the stapler. List as many specific enhancements as you can that would make it better. You may change features, materials, mechanisms, interfaces, or add/remove parts. Do not list new uses; stay focused on improvements to the object itself. For each idea, add enough detail so someone could build or test it.",
                max_list_items=12
            ),
            
            # Task 7: Creative writing
            QuestionFreeText(
                question_name="creative_story",
                question_text="Write a creative short story (3-5 paragraphs) based on this prompt: 'A mysterious door appears in your neighborhood that wasn't there yesterday. When you open it...' Be imaginative and original."
            )
        ]

    def run(self, intervention=False):
        if intervention:
            controller = self.get_creativity_intervention()
            self.model.parameters["controller"] = controller
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")
            
            if "interventions" in controller:
                for item in controller["interventions"]:
                    if "features" in item and "features" in item["features"]:
                        for feature_data in item["features"]["features"]:
                            features = client.features.search(feature_data["label"], model=self.variant, top_k=1)
                            if features:
                                self.variant.set(features[0], item["value"])
        else:
            self.model.parameters["controller"] = {}
            self.variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

        survey = Survey(self.questions)
        return survey.by(self.agents).by(self.model).run(cache=False)

    def get_creativity_intervention(self):
        # NOTE: Replace these with actual feature UUIDs from Goodfire feature search
        return {'interventions': [{'mode': 'nudge',
   'features': {'features': [{'uuid': '2c83bf952a3a4213b45f098aa8c015e2',
      'label': 'Enabling or empowering creative expression and exploration',
      'index_in_sae': 13142,
      'max_activation_strength': 1}]},
   'value': 0.5}],
 'scopes': [],
 'name': 'controller__45610380',
 'nonzero_strength_threshold': None,
 'min_nudge_entropy': None,
 'max_nudge_entropy': None}

    def analyze_features(self, results, condition_name, use_base=False, question_name=None):
        """Analyze and return feature activations for a question."""
        variant = base_variant if use_base else self.variant
        
        # If no specific question name, use the first question
        if question_name is None:
            question_name = self.questions[0].question_name
        
        try:
            prompts = results.select(f"prompt.{question_name}_user_prompt").to_list()
            responses = results.select(f"generated_tokens.{question_name}_generated_tokens").to_list()
            
            if prompts and responses:
                prompt = prompts[0]["text"] if isinstance(prompts[0], dict) else prompts[0]
                response = responses[0]
                
                inspector = client.features.inspect(
                    [{"role": "user", "content": str(prompt)}, {"role": "assistant", "content": str(response)}],
                    model=variant
                )
                
                # Return top features as a list of dicts
                return [{"label": act.feature.label, "activation": act.activation} 
                        for act in inspector.top(k=10)]
        except Exception as e:
            return [{"error": str(e)}]
    
    def get_response_data(self, results, question_name):
        """Extract answer and comment for a question."""
        answer = results.select(f"answer.{question_name}").to_list()
        comment = results.select(f"comment.{question_name}_comment").to_list()
        return {
            "answer": answer[0] if answer else "No response",
            "comment": comment[0] if comment else "No comment"
        }

In [5]:
# Create agents and model
agents = AgentList([Agent(name=f"Agent_{i}", traits={"id": i}) for i in range(1, 2)])
model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")

In [6]:
# Run experiment
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
results_dir = f"creativity_experiment/results_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print("\n" + "="*80)
print("RUNNING EXPERIMENTS")
print("="*80)

# Run Baseline
print("\n⏳ Running BASELINE (No Steering)...")
baseline_model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")
baseline_exp = CreativityExperiment(agents, baseline_model)
baseline_exp.setup()
baseline_results = baseline_exp.run(intervention=False)
baseline_results.to_csv(f"{results_dir}/baseline.csv")
print("✓ Baseline complete")

# Run Steering
print("\n⏳ Running STEERING (Creativity Enhancement)...")
steering_model = Model("meta-llama/Llama-3.3-70B-Instruct", service_name="goodfire")
steering_exp = CreativityExperiment(agents, steering_model)
steering_exp.setup()
steering_results = steering_exp.run(intervention=True)
steering_results.to_csv(f"{results_dir}/steering.csv")
print("✓ Steering complete")

# List of all question names to analyze
questions = ["brick_uses", "animals_talk", "flying_ability", "books_disappear", 
             "planning_task", "stapler_improvements", "creative_story"]

# Display results organized by question
print("\n\n" + "="*80)
print("COMPREHENSIVE RESULTS BY TASK")
print("="*80)

for q in questions:
    print("\n\n" + "█"*80)
    print(f"  TASK: {q.upper().replace('_', ' ')}")
    print("█"*80)
    
    # Get baseline data
    baseline_data = baseline_exp.get_response_data(baseline_results, q)
    baseline_features = baseline_exp.analyze_features(baseline_results, "BASELINE", use_base=True, question_name=q)
    
    # Get steering data
    steering_data = steering_exp.get_response_data(steering_results, q)
    steering_features = steering_exp.analyze_features(steering_results, "STEERING", use_base=False, question_name=q)
    
    # Display Baseline
    print("\n" + "─"*80)
    print("🔷 BASELINE (No Steering)")
    print("─"*80)
    print("\n📝 Response:")
    print(baseline_data["answer"])
    print("\n💭 Comment:")
    print(baseline_data["comment"])
    print("\n🧠 Top 10 Feature Activations:")
    for i, feat in enumerate(baseline_features, 1):
        if "error" in feat:
            print(f"  ⚠️  Error: {feat['error']}")
        else:
            print(f"  {i:2d}. {feat['label']}: {feat['activation']:.2f}")
    
    # Display Steering
    print("\n" + "─"*80)
    print("🔶 STEERING (Creativity Enhanced)")
    print("─"*80)
    print("\n📝 Response:")
    print(steering_data["answer"])
    print("\n💭 Comment:")
    print(steering_data["comment"])
    print("\n🧠 Top 10 Feature Activations:")
    for i, feat in enumerate(steering_features, 1):
        if "error" in feat:
            print(f"  ⚠️  Error: {feat['error']}")
        else:
            print(f"  {i:2d}. {feat['label']}: {feat['activation']:.2f}")

print("\n\n" + "="*80)
print(f"✓ Results saved to: {results_dir}")
print("="*80)


RUNNING EXPERIMENTS

⏳ Running BASELINE (No Steering)...
✓ Baseline complete

⏳ Running STEERING (Creativity Enhancement)...
✓ Steering complete


COMPREHENSIVE RESULTS BY TASK


████████████████████████████████████████████████████████████████████████████████
  TASK: BRICK USES
████████████████████████████████████████████████████████████████████████████████

────────────────────────────────────────────────────────────────────────────────
🔷 BASELINE (No Steering)
────────────────────────────────────────────────────────────────────────────────

📝 Response:
['You can use a brick as a bookend to keep your books organized and add a touch of industrial chic to your home decor, by painting it in a bold color or leaving it in its natural state to create a unique contrast with the surrounding environment.', 'A brick can be repurposed as a garden marker, where you write the name of each plant on the brick and place it next to the corresponding plant, creating a beautiful and functional way to k

In [7]:
# Summary Analysis: Compare creativity features across all tasks
print("\n\n" + "="*80)
print("SUMMARY: CREATIVITY FEATURE COMPARISON ACROSS TASKS")
print("="*80)

summary_data = []

for q in questions:
    baseline_features = baseline_exp.analyze_features(baseline_results, "BASELINE", use_base=True, question_name=q)
    steering_features = steering_exp.analyze_features(steering_results, "STEERING", use_base=False, question_name=q)
    
    # Get top creativity-related feature
    creativity_baseline = next((f for f in baseline_features if 'creativ' in f.get('label', '').lower()), None)
    creativity_steering = next((f for f in steering_features if 'creativ' in f.get('label', '').lower()), None)
    
    summary_data.append({
        'Task': q.replace('_', ' ').title(),
        'Baseline Top Feature': baseline_features[0].get('label', 'N/A')[:50] + '...' if baseline_features else 'N/A',
        'Baseline Top Activation': baseline_features[0].get('activation', 0) if baseline_features else 0,
        'Steering Top Feature': steering_features[0].get('label', 'N/A')[:50] + '...' if steering_features else 'N/A',
        'Steering Top Activation': steering_features[0].get('activation', 0) if steering_features else 0,
        'Creativity Feature Boost': (creativity_steering.get('activation', 0) - creativity_baseline.get('activation', 0)) 
                                     if creativity_baseline and creativity_steering else 'N/A'
    })

# Create DataFrame for easy viewing
summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

print("\n\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)
print("\n1. Tasks where steering had the strongest creativity boost:")
valid_boosts = [(row['Task'], row['Creativity Feature Boost']) 
                for _, row in summary_df.iterrows() 
                if isinstance(row['Creativity Feature Boost'], (int, float))]
if valid_boosts:
    sorted_boosts = sorted(valid_boosts, key=lambda x: x[1], reverse=True)[:3]
    for i, (task, boost) in enumerate(sorted_boosts, 1):
        print(f"   {i}. {task}: +{boost:.2f} activation")

print("\n2. Average activation increase across all tasks:")
avg_boost = summary_df[summary_df['Creativity Feature Boost'] != 'N/A']['Creativity Feature Boost'].mean()
print(f"   Average: +{avg_boost:.2f}")

print("\n3. Strongest overall activations:")
all_activations = []
for _, row in summary_df.iterrows():
    all_activations.append((row['Task'], 'Baseline', row['Baseline Top Activation']))
    all_activations.append((row['Task'], 'Steering', row['Steering Top Activation']))
sorted_acts = sorted(all_activations, key=lambda x: x[2], reverse=True)[:5]
for i, (task, condition, activation) in enumerate(sorted_acts, 1):
    print(f"   {i}. {task} ({condition}): {activation:.2f}")

print("\n" + "="*80)




SUMMARY: CREATIVITY FEATURE COMPARISON ACROSS TASKS


                Task                                  Baseline Top Feature  Baseline Top Activation                                  Steering Top Feature  Steering Top Activation Creativity Feature Boost
          Brick Uses The assistant is generating lists of creative alte...                      223 Enabling or empowering creative expression and exp...                      523                      300
        Animals Talk   Ethical debates around animal welfare and rights...                      253 Enabling or empowering creative expression and exp...                      357                      N/A
      Flying Ability Descriptions of graceful aerial movement and soari...                       70 Enabling or empowering creative expression and exp...                      241                      N/A
     Books Disappear Technical writing patterns explaining function or ...                      177 Enabling or empowering creat